In [3]:
import cutlass
import cutlass.cute as cute


In [5]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, message="CUDA_TOOLKIT_PATH environment variable is not set.*")

In [2]:
@cute.kernel
def hello_kernel():
    tidx, _, _ = cute.arch.thread_idx()
    if tidx == 0:
        cute.printf("Hello from GPU")

@cute.jit
def hello_world():
    cutlass.cuda.initialize_cuda_context()
    hello_kernel().launch(grid=(1, 1, 1), block=(32, 1, 1))

In [ ]:
compiled = cute.compile(hello_world)
compiled()

/home/warmonkeys/miniconda3/envs/cutedsl/lib/python3.12/site-packages/nvidia_cutlass_dsl/python_packages/cutlass/base_dsl/dsl.py:412: UserWarning: CUDA_TOOLKIT_PATH environment variable is not set. Cannot set toolkitPath.
  warnings.warn(message, UserWarning)


0

Hello from GPU


In [4]:
@cute.jit
def print_demo(a: cutlass.Int32, b: cutlass.Constexpr[int]):
    print("static a:", a)   # => ? (dynamic)
    print("static b:", b)   # => 2
    cute.printf("dynamic a: {}", a)
    cute.printf("dynamic b: {}", b)
    layout = cute.make_layout((a, b))
    print("static layout:", layout)       # (?,2):(1,?)
    cute.printf("dynamic layout: {}", layout)  # (8,2):(1,8)

In [5]:
print_demo(cutlass.Int32(8), 2)

static a: ?
static b: 2
static layout: (?,2):(1,?)
dynamic a: 8
dynamic b: 2
dynamic layout: (8,2):(1,8)


In [6]:
@cute.jit
def dtypes():
    a = cutlass.Int32(42)
    b = a.to(cutlass.Float32)
    c = b + 0.5
    d = c.to(cutlass.Int32)
    cute.printf("a={}, b={}, c={}, d={}", a, b, c, d)

dtypes()

a=42, b=42.000000, c=42.500000, d=42


In [1]:
import torch
from cutlass.cute.runtime import from_dlpack

In [4]:
@cute.jit
def tensor_demo(t: cute.Tensor):
    cute.printf("t[0,0] = {}", t[0, 0])
    sub = t[(None, 0)]   # First row view
    frag = cute.make_fragment(sub.layout, sub.element_type)
    frag.store(sub.load())
    cute.print_tensor(frag)

arr = torch.arange(0, 12, dtype=torch.float32).reshape(3, 4)
tensor_demo(from_dlpack(arr))

t[0,0] = 0.000000
tensor(raw_ptr(0x00007ffe5973fa80: f32, rmem, align<32>) o (3):(4), data=
       [ 0.000000, ],
       [ 4.000000, ],
       [ 8.000000, ])


/home/warmonkeys/miniconda3/envs/cutedsl/lib/python3.12/site-packages/nvidia_cutlass_dsl/python_packages/cutlass/base_dsl/dsl.py:412: UserWarning: CUDA_TOOLKIT_PATH environment variable is not set. Cannot set toolkitPath.
  warnings.warn(message, UserWarning)


In [5]:
@cute.jit
def layout_stride_demo(M: cutlass.Int32, N: cutlass.Int32):
    row_major = cute.make_layout((M, N), stride=(N, cutlass.Int32(1)))
    col_major = cute.make_layout((M, N), stride=(cutlass.Int32(1), M))
    print("static row-major:", row_major)
    print("static col-major:", col_major)
    cute.printf("dynamic row-major: {}", row_major)
    cute.printf("dynamic col-major: {}", col_major)

layout_stride_demo(cutlass.Int32(4), cutlass.Int32(3))

static row-major: (?,?):(?,?)
static col-major: (?,?):(?,?)
dynamic row-major: (4,3):(3,1)
dynamic col-major: (4,3):(1,4)


In [9]:
@cute.jit
def slicing_examples(t: cute.Tensor):
    # Scalar access
    cute.printf("t[1,2] = {}", t[1, 2])

    # Entire second row (shape: (N,)) using (None, row_index)
    row = t[(None, 1)]
    row_frag = cute.make_fragment(row.layout, row.element_type)
    row_frag.store(row.load())
    print("Second row:")
    cute.print_tensor(row_frag)

    # Entire third column (shape: (M,)) using (col_index, None)
    col = t[(2, None)]
    col_frag = cute.make_fragment(col.layout, col.element_type)
    col_frag.store(col.load())
    print("Third column:")
    cute.print_tensor(col_frag)

    # Printing the first row directly (*t[2] == *t[2, 0])
    cute.printf(
        "t[2] = {} (equivalent to t[{}])",
        t[2],
        cute.make_identity_tensor(t.layout.shape)[2]
    )
    cute.printf(cute.make_identity_tensor(t.layout.shape))

# 4x3 example tensor
arr = torch.arange(12, dtype=torch.float32).reshape(4, 3)
slicing_examples(from_dlpack(arr))

Second row:
Third column:
t[1,2] = 5.000000
tensor(raw_ptr(0x00007ffe5973fa80: f32, rmem, align<32>) o (4):(3), data=
       [ 1.000000, ],
       [ 4.000000, ],
       [ 7.000000, ],
       [ 10.000000, ])
tensor(raw_ptr(0x00007ffe5973fa60: f32, rmem, align<32>) o (3):(1), data=
       [ 6.000000, ],
       [ 7.000000, ],
       [ 8.000000, ])
t[2] = 6.000000 (equivalent to t[(2,0)])
(0,0) o (4,3):(1@0,1@1)
